# Tools and Routing

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [2]:
from langchain_core.tools import tool

In [3]:
@tool
def search(query: str) -> str:
    """Search for weather online"""
    return "42f"

In [4]:
search.name

'search'

In [5]:
search.description

'Search for weather online'

In [6]:
search.args

{'query': {'title': 'Query', 'type': 'string'}}

In [7]:
from pydantic import BaseModel, Field
class SearchInput(BaseModel):
    query: str = Field(description="Thing to search for")


In [8]:
@tool(args_schema=SearchInput)
def search(query: str) -> str:
    """Search for the weather online."""
    return "42f"

In [9]:
search.args

{'query': {'description': 'Thing to search for',
  'title': 'Query',
  'type': 'string'}}

In [10]:
search.invoke("sf")

'42f'

In [78]:
import requests
from pydantic import BaseModel, Field
import datetime

# Define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> str:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    current_utc_time = datetime.datetime.now(datetime.timezone.utc)
    time_list = [datetime.datetime.fromisoformat(time_str).replace(tzinfo=datetime.timezone.utc) for time_str in results['hourly']['time']]
    temperature_list = results['hourly']['temperature_2m']

    closest_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))
    current_temperature = temperature_list[closest_time_index]

    return f'The current temperature is {current_temperature}°C'

In [79]:
get_current_temperature.name

'get_current_temperature'

In [80]:
get_current_temperature.description

'Fetch current temperature for given coordinates.'

In [81]:
get_current_temperature.args

{'latitude': {'description': 'Latitude of the location to fetch weather data for',
  'title': 'Latitude',
  'type': 'number'},
 'longitude': {'description': 'Longitude of the location to fetch weather data for',
  'title': 'Longitude',
  'type': 'number'}}

In [82]:
from langchain_core.utils.function_calling import convert_to_openai_tool

In [83]:
convert_to_openai_tool(get_current_temperature)

{'type': 'function',
 'function': {'name': 'get_current_temperature',
  'description': 'Fetch current temperature for given coordinates.',
  'parameters': {'properties': {'latitude': {'description': 'Latitude of the location to fetch weather data for',
     'type': 'number'},
    'longitude': {'description': 'Longitude of the location to fetch weather data for',
     'type': 'number'}},
   'required': ['latitude', 'longitude'],
   'type': 'object'}}}

In [84]:
get_current_temperature.invoke({"latitude": 37.7749, "longitude": -122.4194})

'The current temperature is 12.9°C'

In [18]:
import wikipedia
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries."""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [19]:
search_wikipedia.name

'search_wikipedia'

In [20]:
search_wikipedia.description

'Run Wikipedia search and get page summaries.'

In [21]:
convert_to_openai_tool(search_wikipedia)

{'type': 'function',
 'function': {'name': 'search_wikipedia',
  'description': 'Run Wikipedia search and get page summaries.',
  'parameters': {'properties': {'query': {'type': 'string'}},
   'required': ['query'],
   'type': 'object'}}}

In [24]:
search_wikipedia.invoke({"query": "langchain"})

'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain\'s use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n\n\nPage: Vector database\nSummary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).\nVector embeddings are mathematical representations of data in a high-dime

In [27]:
from langchain_classic.chains.openai_functions.openapi import openapi_spec_to_openai_fn
from langchain_community.utilities.openapi import OpenAPISpec

In [28]:
text = """
{
  "openapi": "3.0.0",
  "info": {
    "version": "1.0.0",
    "title": "Swagger Petstore",
    "license": {
      "name": "MIT"
    }
  },
  "servers": [
    {
      "url": "http://petstore.swagger.io/v1"
    }
  ],
  "paths": {
    "/pets": {
      "get": {
        "summary": "List all pets",
        "operationId": "listPets",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "limit",
            "in": "query",
            "description": "How many items to return at one time (max 100)",
            "required": false,
            "schema": {
              "type": "integer",
              "maximum": 100,
              "format": "int32"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "A paged array of pets",
            "headers": {
              "x-next": {
                "description": "A link to the next page of responses",
                "schema": {
                  "type": "string"
                }
              }
            },
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pets"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      },
      "post": {
        "summary": "Create a pet",
        "operationId": "createPets",
        "tags": [
          "pets"
        ],
        "responses": {
          "201": {
            "description": "Null response"
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    },
    "/pets/{petId}": {
      "get": {
        "summary": "Info for a specific pet",
        "operationId": "showPetById",
        "tags": [
          "pets"
        ],
        "parameters": [
          {
            "name": "petId",
            "in": "path",
            "required": true,
            "description": "The id of the pet to retrieve",
            "schema": {
              "type": "string"
            }
          }
        ],
        "responses": {
          "200": {
            "description": "Expected response to a valid request",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Pet"
                }
              }
            }
          },
          "default": {
            "description": "unexpected error",
            "content": {
              "application/json": {
                "schema": {
                  "$ref": "#/components/schemas/Error"
                }
              }
            }
          }
        }
      }
    }
  },
  "components": {
    "schemas": {
      "Pet": {
        "type": "object",
        "required": [
          "id",
          "name"
        ],
        "properties": {
          "id": {
            "type": "integer",
            "format": "int64"
          },
          "name": {
            "type": "string"
          },
          "tag": {
            "type": "string"
          }
        }
      },
      "Pets": {
        "type": "array",
        "maxItems": 100,
        "items": {
          "$ref": "#/components/schemas/Pet"
        }
      },
      "Error": {
        "type": "object",
        "required": [
          "code",
          "message"
        ],
        "properties": {
          "code": {
            "type": "integer",
            "format": "int32"
          },
          "message": {
            "type": "string"
          }
        }
      }
    }
  }
}
"""

In [30]:
# spec = OpenAPISpec.from_text(text)
import json
spec = json.loads(text)

In [32]:
# Build OpenAI tool definitions directly from the OpenAPI spec
pet_openai_tools = []
for path, methods in spec["paths"].items():
    for method, details in methods.items():
        params = details.get("parameters", [])
        properties = {}
        required = []
        for p in params:
            properties[p["name"]] = {
                "type": p["schema"]["type"],
                "description": p.get("description", ""),
            }
            if p.get("required"):
                required.append(p["name"])

        pet_openai_tools.append({
            "type": "function",
            "function": {
                "name": details["operationId"],
                "description": details.get("summary", ""),
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": required,
                },
            },
        })

In [33]:
pet_openai_tools

[{'type': 'function',
  'function': {'name': 'listPets',
   'description': 'List all pets',
   'parameters': {'type': 'object',
    'properties': {'limit': {'type': 'integer',
      'description': 'How many items to return at one time (max 100)'}},
    'required': []}}},
 {'type': 'function',
  'function': {'name': 'createPets',
   'description': 'Create a pet',
   'parameters': {'type': 'object', 'properties': {}, 'required': []}}},
 {'type': 'function',
  'function': {'name': 'showPetById',
   'description': 'Info for a specific pet',
   'parameters': {'type': 'object',
    'properties': {'petId': {'type': 'string',
      'description': 'The id of the pet to retrieve'}},
    'required': ['petId']}}}]

In [34]:
from langchain_openai import ChatOpenAI

In [35]:
# Wrap old-style function defs into tools format for bind_tools
# pet_openai_tools = [{"type": "function", "function": f} for f in pet_openai_functions]
model = ChatOpenAI(temperature=0).bind_tools(pet_openai_tools)

In [36]:
model.invoke("what are three pets names")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 108, 'total_tokens': 122, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSc1gsyRehg96dKLTDEFACKRiTqYm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d70aa-aab6-7d01-91f0-5b98a43a2d52-0', tool_calls=[{'name': 'listPets', 'args': {'limit': 3}, 'id': 'call_8xFifkDjdrDyOjxpxoVjoekm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 108, 'output_tokens': 14, 'total_tokens': 122, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [37]:
model.invoke("tell me about pet with id 42")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 111, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSc1jNQTxSQMe2zzSu4ZxjA35k8aC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d70aa-b916-70a2-8d6e-cec3ac6662b8-0', tool_calls=[{'name': 'showPetById', 'args': {'petId': '42'}, 'id': 'call_0PjP1RbDGeSVgRZSM6f3S60M', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 16, 'total_tokens': 127, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Routing

In lesson 3, we show an example of function calling deciding between two candidate functions.

Given our tools above, let's format these as OpenAI functions and show this same behavior.

In [87]:
model = ChatOpenAI(temperature=0).bind_tools(
    [search_wikipedia, get_current_temperature]
)

In [88]:
model.invoke("what is the weather in sf right now")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 105, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DScUVqu8HLrIKtaAdRriQGq3GjAzD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d70c5-f057-78f0-9247-271238409db3-0', tool_calls=[{'name': 'search_wikipedia', 'args': {'query': 'San Francisco'}, 'id': 'call_LkGYoT3dY0BQIdqAuK1y2DEa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 105, 'output_tokens': 16, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [89]:
model.invoke("what is langchain")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 101, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DScUWk9bsnohOs6nts3tFWqvTTlph', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d70c5-f594-7793-9982-244f3105b760-0', tool_calls=[{'name': 'search_wikipedia', 'args': {'query': 'Langchain'}, 'id': 'call_Tlgsjr9SGLS9YITM4wirVIfL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 101, 'output_tokens': 16, 'total_tokens': 117, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [90]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    ("user", "{input}"),
])
chain = prompt | model

In [91]:
chain.invoke({"input": "what is the weather in sf right now"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 113, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DScUaBE1HTFdmC3KIZcae15bGdCmK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d70c6-0133-7951-9200-a969dce86a9e-0', tool_calls=[{'name': 'search_wikipedia', 'args': {'query': 'San Francisco'}, 'id': 'call_hczQdIAEfca1v6mh0BtU0d2r', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 113, 'output_tokens': 16, 'total_tokens': 129, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [92]:
from langchain_classic.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

In [93]:
chain = prompt | model | OpenAIToolsAgentOutputParser()

In [94]:
# OpenAIToolsAgentOutputParser returns list[AgentAction] or AgentFinish
result = chain.invoke({"input": "what is the weather in sf right now"})

In [95]:
# result is a list of AgentActions when tools are called
type(result), type(result[0])

(list, langchain_classic.agents.output_parsers.tools.ToolAgentAction)

In [96]:
result[0].tool

'search_wikipedia'

In [97]:
result[0].tool_input

{'query': 'San Francisco'}

In [98]:
get_current_temperature.invoke(result[0].tool_input)

ValidationError: 2 validation errors for OpenMeteoInput
latitude
  Field required [type=missing, input_value={'query': 'San Francisco'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
longitude
  Field required [type=missing, input_value={'query': 'San Francisco'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [99]:
result = chain.invoke({"input": "hi!"})

In [100]:
type(result)

langchain_core.agents.AgentFinish

In [101]:
result.return_values

{'output': 'Hello! How can I assist you today?'}

In [102]:
from langchain_core.agents import AgentFinish
def route(result):
    if isinstance(result, AgentFinish):
        return result.return_values['output']
    else:
        # result is a list of AgentActions; execute the first tool call
        action = result[0]
        tools = {
            "search_wikipedia": search_wikipedia,
            "get_current_temperature": get_current_temperature,
        }
        return tools[action.tool].invoke(action.tool_input)

In [103]:
chain = prompt | model | OpenAIToolsAgentOutputParser() | route

In [104]:
result = chain.invoke({"input": "What is the weather in san francisco right now?"})

In [105]:
result

"Page: San Francisco\nSummary: San Francisco, officially the City and County of San Francisco, is the fourth-most populous city in California and the 17th-most populous in the United States, with a population of 826,079 in 2025. Among U.S. cities with a population of 300,000 or more, San Francisco is ranked first by per capita income, second by population density, and sixth by aggregate income as of 2023. Some 4.6 million residents live in the city's metropolitan statistical area, which is the 13th-largest in the United States. Around 9.2 million live in the San Jose–San Francisco–Oakland combined statistical area, the fifth-largest in the United States.\nPrior to European settlement, San Francisco was inhabited by the Yelamu Ohlone. On June 29, 1776, settlers from New Spain established the Presidio of San Francisco at the Golden Gate, and the Mission San Francisco de Asís a few miles away, both named for Francis of Assisi. The California gold rush of 1849 brought rapid growth, making 

In [106]:
result = chain.invoke({"input": "What is langchain?"})

In [107]:
result

'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain\'s use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n\n\nPage: Vector database\nSummary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).\nVector embeddings are mathematical representations of data in a high-dime

In [108]:
chain.invoke({"input": "hi!"})

'Hello! How can I assist you today?'